# Sold Weeks 4-5

In [1]:
import pandas as pd

sold = pd.read_csv("../../IDX_Data/sold_with_rates.csv")
print("Initial shape:", sold.shape)
sold.head()

/var/folders/hx/s9px6scd2pl5dqfp0pldbdgc0000gn/T/ipykernel_54696/3013951622.py:3: DtypeWarning: Columns (0: BuyerAgentAOR, 1: ListAgentAOR, 2: WaterfrontYN, 3: ListAgentEmail, 4: FireplaceYN, 5: OriginatingSystemName, 6: OriginatingSystemSubName, 7: BuyerAgencyCompensationType, 8: latfilled, 9: lonfilled) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv("../../IDX_Data/sold_with_rates.csv")


Initial shape: (397172, 86)


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,OriginatingSystemName,OriginatingSystemSubName,BuyerAgencyCompensationType,BuyerAgencyCompensation,latfilled,lonfilled,year_month,rate_30yr_fixed
0,Mlslistings,Mlslistings,"Carpet,Tile,Wood",True,NaN,NaN,False,499000.0,551985747,jwachter@cbnorcal.com,...,NaN,NaN,CRMLS,CRMLS_MLSL,NaN,NaN,NaN,NaN,2024-01,6.6425
1,SanDiego,SanDiego,NaN,False,NaN,NaN,False,759900.0,522107581,mdarwich12@gmail.com,...,NaN,NaN,CRMLS,CRMLS_SAND,NaN,NaN,NaN,NaN,2024-01,6.6425
2,SanDiego,SanDiego,NaN,False,NaN,NaN,False,739900.0,510919001,mdarwich12@gmail.com,...,NaN,NaN,CRMLS,CRMLS_SAND,NaN,NaN,NaN,NaN,2024-01,6.6425
3,Mlslistings,Mlslistings,NaN,False,NaN,NaN,NaN,NaN,1079166779,davidmartz@compass.com,...,13504.0,NaN,CRMLS,CRMLS_MLSL,NaN,NaN,NaN,NaN,2024-01,6.6425
4,Southland,Southland,NaN,False,NaN,NaN,False,1890500.0,1075037759,karen.klein@theagencyre.com,...,17873.0,NaN,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN,2024-01,6.6425


In [2]:
date_cols = [
    'CloseDate',
    'PurchaseContractDate',
    'ListingContractDate',
    'ContractStatusChangeDate'
]

for col in date_cols:
    if col in sold.columns:
        sold[col] = pd.to_datetime(sold[col], errors='coerce')

sold[date_cols].head()

,CloseDate,PurchaseContractDate,ListingContractDate,ContractStatusChangeDate
0,2024-01-26,2023-11-22,2021-10-06,2024-01-26
1,2024-01-05,2021-06-30,2021-03-08,2024-01-05
2,2024-01-05,2021-11-18,2021-03-08,2024-01-05
3,2024-01-30,2024-08-05,2024-01-30,2024-01-30
4,2024-01-29,2024-01-29,2024-01-29,2024-01-29


In [3]:
missing_pct = sold.isnull().mean() * 100
high_missing_cols = missing_pct[missing_pct > 90].index.tolist()

print("Columns >90% missing:")
print(high_missing_cols)

Columns >90% missing:
['WaterfrontYN', 'BasementYN', 'FireplacesTotal', 'AboveGradeFinishedArea', 'TaxAnnualAmount', 'BuilderName', 'TaxYear', 'BuildingAreaTotal', 'ElementarySchoolDistrict', 'CoBuyerAgentFirstName', 'BelowGradeFinishedArea', 'BusinessType', 'CoveredSpaces', 'LotSizeDimensions', 'MiddleOrJuniorSchoolDistrict', 'OriginatingSystemName', 'OriginatingSystemSubName']


In [4]:
before_cols = sold.shape[1]
sold = sold.drop(columns=high_missing_cols)
after_cols = sold.shape[1]

print("Columns before:", before_cols)
print("Columns after:", after_cols)

Columns before: 86
Columns after: 69


In [5]:
numeric_cols = [
    'ClosePrice',
    'LivingArea',
    'DaysOnMarket',
    'BedroomsTotal',
    'BathroomsTotalInteger'
]

for col in numeric_cols:
    if col in sold.columns:
        sold[col] = pd.to_numeric(sold[col], errors='coerce')

sold[numeric_cols].dtypes

ClosePrice               float64
LivingArea               float64
DaysOnMarket               int64
BedroomsTotal            float64
BathroomsTotalInteger    float64
dtype: object

In [6]:
before_rows = sold.shape[0]
sold = sold[
    (sold['ClosePrice'] > 0) &
    (sold['LivingArea'] > 0) &
    (sold['DaysOnMarket'] >= 0) &
    (sold['BedroomsTotal'] >= 0) &
    (sold['BathroomsTotalInteger'] >= 0)
]
after_rows = sold.shape[0]

print("Rows before:", before_rows)
print("Rows after:", after_rows)

Rows before: 397172
Rows after: 396693


In [7]:
sold['listing_after_close_flag'] = (
    sold['ListingContractDate'] > sold['CloseDate']
)

sold['purchase_after_close_flag'] = (
    sold['PurchaseContractDate'] > sold['CloseDate']
)

sold['negative_timeline_flag'] = (
    (sold['ListingContractDate'] > sold['PurchaseContractDate']) |
    (sold['PurchaseContractDate'] > sold['CloseDate'])
)

In [8]:
print("Listing after close:", sold['listing_after_close_flag'].sum())
print("Purchase after close:", sold['purchase_after_close_flag'].sum())
print("Negative timeline:", sold['negative_timeline_flag'].sum())

Listing after close: 58
Purchase after close: 240
Negative timeline: 499


In [9]:
missing_coords = sold['Latitude'].isnull() | sold['Longitude'].isnull()
zero_coords = (sold['Latitude'] == 0) | (sold['Longitude'] == 0)
positive_longitude = sold['Longitude'] > 0

print("Missing coords:", missing_coords.sum())
print("Zero coords:", zero_coords.sum())
print("Positive longitude:", positive_longitude.sum())

Missing coords: 15813
Zero coords: 25
Positive longitude: 29


In [10]:
sold.to_csv("sold_cleaned_week4_5.csv", index=False)
print("Cleaned SOLD dataset saved.")

Cleaned SOLD dataset saved.


### Data Cleaning Summary

- Converted date fields to datetime format
- Removed columns with more than 90% missing values
- Ensured numeric fields were properly typed
- Removed invalid numeric values (e.g., negative or zero values)
- Created date consistency flags
- Performed geographic data quality checks

### Validation Summary

- Row counts before and after filtering were tracked
- Date inconsistencies were identified using flag variables
- Geographic anomalies were flagged (missing or invalid coordinates)
- Final dataset is cleaned and ready for analysis